In [ ]:
# =============================================================================
# FUMD-AI Training & Postprocessing Workflow -- Step 4: Aggregate training metrics
# =============================================================================
# Step:         4 of 6
# Summary:      Compare Step 3 metrics across evaluated datasets and/or model variants -- tables and plots, from structured data.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-training-postprocessing-workflow
# Version:      0.1.1
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# =============================================================================

# Step 4 — Aggregate training metrics

Part of the **FUMD-AI training & postprocessing workflow** (step 4 of 6,
first postprocessing step).

**Purpose.** Read every `metrics_<run_label>.json` Step 3 wrote (across one
or more Step 2 runs -- e.g. different hyperparameters or architecture
variants you want to compare) and build the same kind of comparison
tables/plots the original `leer_metricas.ipynb` did (per-output loss/
accuracy, cross-dataset comparison) -- but reading structured JSON/CSV
directly (`metrics_io.discover_metrics`/`to_long_dataframe`) instead of
regex-scraping printed `.txt` logs.

**Input:** `RUN_DIRS` -- one or more Step 2/3 output directories.
`VARIANT_LABELS` optionally names each one (e.g. `"bahdanau"` vs a future
alternative) for the plot legend; defaults to each directory's name.

**Outputs**, written under `COMPARISON_OUTPUT_DIR`:
- `metrics_long.csv` -- every run's metrics in one long-format table
- `comparison_accuracy_by_output.png`, `comparison_top2_accuracy_by_output.png`


In [ ]:
import os
import sys

import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # `jupyter execute` sets the kernel's cwd to this notebook's OWN
                     # directory (see nbclient's cli.py), not the repo root -- normalize
                     # back so `sys.path.insert(0, "src")` and every relative default
                     # below resolve the same way whether this notebook is run standalone
                     # (jupyter execute notebooks/step_N...ipynb, e.g. via slurm/*.sbatch)
                     # or chained by run_pipeline.ipynb (whose own cwd is already the repo
                     # root, and which additionally sets cwd explicitly -- see
                     # fumd_training_workflow.notebook_runner.run_step).

sys.path.insert(0, "src")
from fumd_training_workflow.metrics_io import discover_metrics, to_long_dataframe

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ---- Parameters ----
RUN_DIRS = ["pipeline_run"]     # one or more Step 2/3 output directories to compare
VARIANT_LABELS = None            # None -> use each directory's basename

COMPARISON_OUTPUT_DIR = "metrics_comparison"

In [ ]:
# ---- Environment variable overrides (optional) ----
# `jupyter execute` has no -p/parameter-injection flag (unlike papermill), so
# each parameter above can also be set via an environment variable of the same
# name instead of hand-editing this cell, e.g.:
#   DATASET_PATH=/abs/path/dataset_labeled_w3.csv OUTPUT_DIR=/abs/path/pipeline_run \
#   jupyter execute notebooks/step_4_aggregate_training_metrics.ipynb
# See fumd_training_workflow.notebook_runner.env_override's docstring for exact
# value-parsing rules. Unset variables leave the parameters cell's own default above untouched.
from fumd_training_workflow.notebook_runner import env_override

RUN_DIRS = env_override('RUN_DIRS', RUN_DIRS)
VARIANT_LABELS = env_override('VARIANT_LABELS', VARIANT_LABELS)
COMPARISON_OUTPUT_DIR = env_override('COMPARISON_OUTPUT_DIR', COMPARISON_OUTPUT_DIR)

## 1. Load every run's structured metrics

In [ ]:
variant_labels = VARIANT_LABELS or [os.path.basename(os.path.normpath(d)) for d in RUN_DIRS]
assert len(variant_labels) == len(RUN_DIRS)

frames = []
for run_dir, variant in zip(RUN_DIRS, variant_labels):
    payloads = discover_metrics(run_dir)
    print(f"{run_dir} ({variant}): {len(payloads)} metrics file(s) -> "
          f"{[p['run_label'] for p in payloads]}")
    frames.append(to_long_dataframe(payloads, variant_label=variant))

long_df = pd.concat(frames, ignore_index=True)
os.makedirs(COMPARISON_OUTPUT_DIR, exist_ok=True)
long_df.to_csv(os.path.join(COMPARISON_OUTPUT_DIR, "metrics_long.csv"), index=False)
long_df.head()

## 2. Comparison plots

In [ ]:
def comparison_plot(metric_name, filename):
    sub = long_df[long_df["metric"] == metric_name]
    if sub.empty:
        print(f"no rows for metric={metric_name}, skipping")
        return
    g = sns.catplot(
        data=sub, kind="bar",
        x="future_step", y="value", hue="variant" if sub["variant"].nunique() > 1 else "run_label",
        col="run_label" if sub["variant"].nunique() <= 1 else None,
        col_wrap=3 if sub["variant"].nunique() <= 1 else None,
        height=4, aspect=1.2, palette="viridis",
    )
    g.set_axis_labels("Forecast step (s ahead)", metric_name.replace("_", " ").title())
    g.fig.suptitle(f"{metric_name.replace('_', ' ').title()} by forecast step", y=1.03)
    out_path = os.path.join(COMPARISON_OUTPUT_DIR, filename)
    g.savefig(out_path, dpi=150, bbox_inches="tight")
    print("saved", out_path)


comparison_plot("accuracy", "comparison_accuracy_by_output.png")
comparison_plot("top2_accuracy", "comparison_top2_accuracy_by_output.png")
comparison_plot("f1", "comparison_f1_by_output.png")

## 3. Summary table

In [ ]:
summary = (
    long_df.groupby(["variant", "run_label", "metric"], dropna=False)["value"]
    .mean()
    .unstack("metric")
    .reset_index()
    .sort_values(["variant", "run_label"])
)
summary_path = os.path.join(COMPARISON_OUTPUT_DIR, "metrics_summary.csv")
summary.to_csv(summary_path, index=False)
print(f"saved {summary_path}")
summary